# King Domino Projekt - Overblik

### Arkitekturoversigt
Projektets formål er at digitalisere og point-score fysiske King Domino-spilleplader. Projektet består af fem hovedfaser:
1. **Gitteropdeling (Grid Splitting):** 
    - Opdeling af hele pladebilleder i et 5x5 logisk gitter af terræn-tiles.
2. **Feature-ekstraktion:** 
    - Behandling af tærren-tiles for at udtrække features til en 20-værdi vektor (HSV-histogram logik)
    - Gemme 20-værdi vektorerne i en csv-fil til modeltræning/analyse.
3. **Tile-multiklassificering:** 
    - Identificering af tile-terræntypen (Water, Field, Grass, Forest, Mine, Swamp, Home, Empty). 
    - Altså en multi-class terrænklassificering udført med enten SVM eller kNN
4. **Krone-detektion:** 
    - Tælle antallet af score-multiplikatorer (kroner) til stede på hver klassificeret terræn-tile ved at bruge template matching (fx med `high_res_crown.png` / `low_res_crown.png`). 
    - Behandle gråtone/blobs
    - Håndtering af gennemsigtige "Don't care"-pixels.
5. **Spillogik og Scoreberegning:** 
    - Evaluering af 2D-arrays, der repræsenterer spillepladernes 5x5-gitter.
    - Hver spilleplade (fx med Breadth-First Search algoritme) til at beregne den endelige score for regioner (forbundne terræn-tiles).
    - Baseret på Kingdomino-reglerne(fx 6 forbundne vand-tiles * 2 kroner = 12 point).
6. **Ground Truths(GT):**
    - GT_BoardPointScore (Done!)
    - GT_BoardTileMatrix (Done!)
    - GT_CrownsPerBoard (Done!)
    - GT_CrownsPerTile



### Nuværende status

| Modul | Status | Beskrivelse |
| :--- | :---: | :--- |
| `board_split.py` | 🟢 | Deler spilleplader op i et 5x5 tile-gitter. |
| `feature_extraction.py` | 🟢 | Udtrækker 20-værdi vektor HSV-histogrammer fra tiles og gemmer data i et dataset (`features.csv`). |
| `extract_tile_feature_vector.py` | 🟡 | Indlæser billeder, opdeler tiles og udfører feature-ekstraktion |
| `classify_tile.py` | 🟢 | Implementerer `TileClassifier` ved hjælp af en Scikit-learn k-Nearest Neighbors (kNN) model trænet på `features.csv`. |
| `detect_crown.py` | 🟡 | (`CrownDetector`). Kræver kontur/form-detektion for at tælle 0-3 kongekroner per tile. |
| `scoring.py` | 🔴 | Stub (`ScoreCalculator`). Kræver BFS-algoritmen til at gange størrelsen af forbundne regioner med antal kroner. |

### Manglende dele
1. **Krone-tællelogik:** Identificering af specifikke gule konturer/former på ikke-tomme tiles for at fungere som multiplikatorer (`detect_crown.py`).
2. **Spilleregel-motor:** Implementering af BFS-gittergennemløb for at gruppere tilstødende matchende terræner og beregne den endelige score (`scoring.py`).
3. **End-to-End Integration:** Modificering af `extract_tile_feature_vector.py` til at bruge klassificeringsmodellerne og sende det resulterende 5x5 gitter af objekter til scoringsmotoren.

### Anbefalet næste skridt
**Implementer krone-detektion (`detect_crown.py`)**
Nu hvor vi kan ekstrahere features og klassificere terræntypen for hver tile med en ML-model, er den næste komponent at bygge krone-detektionen. Dette vil identificere score-multiplikatorer.

**Næste kode der skal genereres:**
Vi bør udfylde logikken indeni `CrownDetector` (`detect_crown.py`) for at detektere og tælle forekomster af gyldne kroner via template matching eller konturdetektion, med højde for potentiel støj / overlappende elementer.

## 1. Dataforbehandling og split-strategi
For at undgå datalækage laves split konsekvent på spilleplade-niveau og ikke på tile-niveau. Parrede billeder (med og uden 3D-objekter) placeres altid i samme split.

* **Samlet antal spilleplader:** 36 unikke spilleplader.
* **Ugyldige data(støj):** Billede 71, 73 og 74 er udeladt, fordi de er dubletter af billede 68, 69 og 70.
* **Dataekstraktion:** ca. 1775 tiles (71 gyldige billeder × 25 felter).

### Endelig split:
* **Hold-out testsæt:** 6 spilleplade-grupper (12 billeder i alt) reserveres til den endelige evaluering for at teste præstation af ML-model.

Test-sæt:
Spilleplader
  - (1,5), (19,23), (25,29), (35,39), (49,53), (67,70)

* **Træning og validering (grouped CV):** De resterende 30 spilleplade-grupper bruges til 5-fold grouped cross-validation.
  * Hver fold indeholder 6 spilleplade-grupper.
  * I hver CV-runde bruges 4 folds til træning og 1 fold til validering.

Fold 1:
Spilleplader 
  - Mixed_data: (4,8), (20,24), (34,38), (42,46), (48,52), (65,72)
  - Uden3dObjekt(clean_data): 3, 20, 34, 42, 48, 65
  - Med3dObjekt(dirty_data): 8, 24, 38, 46, 52, 72

Fold 2:
Spilleplader
(2,6), (18,22), (28,32), (36,40), (51,55), (58,62)

Fold 3:
Spilleplader
(10,14), (11,15), (26,30), (41,44), (57,61), (64,68)

Fold 4:
Spilleplader
(3,7), (17,21), (27,31), (43,47), (50,54), (59,63)

Fold 5:
Spilleplader
(9,13), (12,16), (33,37), (45), (56,60), (66,69)


-
-
-
-
-
-
-
-


## 📈 Projektstatus og Arkitektur (Opdateret)

Projektet er struktureret som en Computer Vision og Machine Learning pipeline til at digitalisere og score Kingdomino brætspil.

### 🏗️ Arkitektur & Dataflow

1.  **Billedbehandling og Segmentering (`board_split.py`, `save_tiles.py`)**: Opdeler det fulde spillepladebillede i et 5x5 grid af 128x128 pixels brikker. Brikkerne gemmes hierarkisk i `KD_tiles` mappen.
2.  **Egenskabsekstraktion (Feature Extraction) (`feature_extraction.py`, `extract_tile_feature_vector.py`)**: Konverterer brikkernes billeder til HSV farverummet. Udtrækker farve-histogrammer (Hue: 10 bins, Saturation: 5 bins, Value: 5 bins), fladgør dem og gemmer i `features.csv` (20-dimensionel feature vektor).
3.  **Klassificering af Terræn (`classify_tile.py`)**: Anvender scikit-learn's k-Nearest Neighbors (kNN) algoritme, trænet på `features.csv`, til at forudsige brikkens terræntype (Water, Field, Grass, Forest, Mine, Swamp, Home, Empty).
4.  **Kronedetektion (`detect_crown.py`, `prepare_crown_template.py`)**: Anvender maskering i HSV farverummet samt kontur-detektion for at tælle antallet af guld-kroner (0-3) på hver brik. Der er også forberedt skabeloner (template matching) til at forbedre robustheden.
5.  **Scoreberegning (`scoring.py`)**: Beregner den endelige pointscore ved hjælp af en graf-søgningsalgoritme (BFS/DFS), der finder sammenhængende terrænområder og ganger områdets størrelse med det samlede antal kroner i området.

### 🟢 Nuværende Fremskridt (Færdiggjort)
*   **Segmentering:** Virker, brættet kan opdeles og brikker udtrækkes.
*   **Feature Extraction:** Histogramudtrækkelse og strukturering i CSV format er implementeret.
*   **Klassifikationsmodel:** En kNN-model til terrænklassificering er opsat.

### 🟡 Under Udvikling
*   **End-to-End integration:** Mangler en overordnet pipeline-script, der binder det hele sammen i et rent flow fra input-billede til output-score.
*   **Kronedetektion:** Dette trin kræver muligvis optimering og robustheds-test (f.eks. ved overgangen fra ren contour-detektion til template matching) for at eliminere false positives og false negatives fra skygger eller brikkens motiver.

### 🔴 Udestående Arbejde (Next Steps)
1.  **Scoring-algoritme (`scoring.py`)**: Implementering af Breadth-First Search (BFS) eller Depth-First Search (DFS) til at traversere 5x5 griddet og summere `områdestørrelse * kroner`.
2.  **Evaluering og Validationsstrategi**: Oprettelse af et formelt test-setup (train/val/test split pr. bræt - for at undgå data leakage). Krydsvalidering af modellens præcision samt end-to-end score fejl-margin (RMS, MAE).
3.  **Hoved-Orchestrator**: Opret et `main.py` (eller en Master Notebook), der trækker et råt billede og udfører alle trin sekventielt: `Billede -> Split -> 25 Brikker -> (Terræn + Kroner) per. Brik -> Scoring Engine -> Endelig Score`.

### 🎓 Akademisk Rigor og Forskningstilgang
*   **Reproducerbarhed**: Sikre alle seed-værdier er sat (`np.random.seed`, etc.), og at `requirements.txt` dækker ethvert biblotek med præcise versioner.
*   **Eksperimentsporing**: Vi bør logge hvornår og hvordan features er genereret, samt modellens præstation med metrics som Precision, Recall og F1-score (i tillæg til den generelle end-to-end nøjagtighed af pointberegningen).

## 🚀 Ekspertanalyse og Næste Skridt (Tilføjet)

Som Senior ML Research Engineer har jeg gennemgået kodestrukturen, ML-pipelinen og projektets nuværende stadie. Jeres arbejde med `feature_extraction.py` og cross-validation opsætningen i `classify_tile.py` viser en stærk forståelse for at forhindre datalækage (grouping by board) og systematisk hyperparameter-tuning.

### 🔍 Identificerede Gaps & Risici
1. **Manglende Main-Orkestrator:** Projektet mangler en samlet `main.py` eller en dedikeret notebook til at køre en fuld end-to-end inferens (Billede $\rightarrow$ Grid $\rightarrow$ Tile Klassificering $\rightarrow$ Krone Detektion $\rightarrow$ Endelig pointberegning).
2. **Scoring Motoren (`scoring.py`):** Denne er i øjeblikket blot en "stub". Det er et kritisk modul for at kunne reproducere Kingdoms-reglerne via Breadth-First Search (BFS) / Depth-First Search (DFS).
3. **Fejlpropagering (Error Propagation):** En klassifikationsfejl på blot én tile, eller en overset krone, kan drastisk ændre den endelige score på brættet. End-to-end metrikker (fx. MAE - Mean Absolute Error på bræt-point) mangler. Det er ikke nok kun at måle tile-klassifikations-accuracy.
4. **Småfejl i repository:** Filen `requiments.txt` har en stavefejl og bør rettes til `requirements.txt` af hensyn til standarder og CI/CD pipelines.

### 🏛️ Foreslået Arkitekturforbedring
Vi bør anvende adskilte ansvarsområder (Separation of Concerns):
- **Computer Vision Modul:** `board_split.py`, `save_tiles.py` (Håndterer kun pixels og geometri).
- **Machine Learning Modul:** `feature_extraction.py`, `classify_tile.py`, `detect_crown.py` (Omdanner pixels til semantisk data: terræntype, krone-antal).
- **Forretningslogik / Rules Engine:** `scoring.py` (Tager et 5x5 array af objekter og returnerer en integer score — totalt agnostisk over for billeddata).
- **Orkestrator:** `evaluate_pipeline.py` (Samler de tre moduler og beregner RMSE/MAE mellem forudsagt score og Ground Truth i `Labels_ground_truth.csv`).

### 🎯 Handlingsplan (Action Items)
Her er den prioriterede rækkefølge for implementering af de manglende dele, som sikrer akademisk stringens og reproducerbarhed:

1. **Trin 1: Implementering af Spilleregler (`scoring.py`)** 
   - Udvikl og enhedstest (unit test) BFS/DFS algoritmen. Den skal kunne tage et syntetisk/perfekt 5x5 grid (fra Ground Truths) og altid udregne den 100% korrekte score. Gør vi dette først, har vi en stensikker evalueringsmekanisme.
2. **Trin 2: Integration & End-to-End Pipeline** 
   - Byg et script, der fodrer predictions fra `classify_tile.py` (og dummy/GT kroner i første omgang) ind i `scoring.py`. Etablér metrikkerne for fejl på bræt-niveau.
3. **Trin 3: Krone-detektion (`detect_crown.py`)**
   - Implemementer farve-maskering (guld/gul i HSV) kombineret med formgenkendelse/template-matching. Evaluer præcisionen særskilt før det puttes ind i den samlede pipeline.
4. **Trin 4: Dokumentation, Omgivelser (Requirements) og Final Tuning**
   - Ret navngivning, fastfrys versionsnumre med `pip freeze > requirements.txt` og opsummer resultaterne.